# Gold Layer — Data Transformation
Build the Gold layer from the Silver market data and reference tables in Book1.xlsx.

Grain of fact_market_snapshot: one row per snapshot_id + product_id + quality + kind.

The notebook creates and saves:
- dim_product
- dim_building
- dim_date
- bridge_product_recipe
- fact_production_capacity
- fact_market_snapshot
Gold outputs are written as separate Parquet directories under data/gold/.

# 1. Project paths and imports

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Locate project root (the directory containing data/)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project directory not found. Run the notebook from inside the project.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_FILE = PROJECT_ROOT / "data" / "source" / "Book1.xlsx"
SILVER_PATH = PROJECT_ROOT / "data" / "silver" / "market"
GOLD_PATH = PROJECT_ROOT / "data" / "gold"

# Make the project's src/ directory importable.
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from simcompanies.schemas.gold import (
    dim_product_schema,
    dim_building_schema,
    dim_date_schema,
    bridge_product_recipe_schema,
    fact_production_capacity_schema,
    fact_market_snapshot_schema,
)

assert SOURCE_FILE.exists(), f"Reference workbook not found: {SOURCE_FILE}"
assert SILVER_PATH.exists(), f"Silver market data not found: {SILVER_PATH}"
GOLD_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Silver path:", SILVER_PATH)
print("Gold path:", GOLD_PATH)


Project root: c:\Projekty\SimCompanies
Silver path: c:\Projekty\SimCompanies\data\silver\market
Gold path: c:\Projekty\SimCompanies\data\gold


In [2]:
import os
import sys
import pyspark

from pathlib import Path
from pyspark.sql import SparkSession

# Environment
pyspark_home = Path(pyspark.__file__).parent

os.environ["SPARK_HOME"] = str(pyspark_home)
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Spark
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("SimCompanies Gold Layer")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Python:", sys.executable)
print("Context stopped:", spark.sparkContext._jsc is None)

Spark: 3.5.1
Python: c:\Users\Prezes\AppData\Local\Programs\Python\Python310\python.exe
Context stopped: False


# 2. Spark session and Silver input

In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("SimCompanies Gold Layer")
    .getOrCreate()
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

df_market = spark.read.parquet(str(SILVER_PATH))

print("Silver rows:", df_market.count())
df_market.printSchema()
df_market.show(5, truncate=False)


def pandas_to_spark(pdf, schema):
    return spark.createDataFrame(pdf.to_dict(orient="records"), schema=schema)


Silver rows: 39208
root
 |-- captured_at: timestamp (nullable = true)
 |-- source_page_id: integer (nullable = true)
 |-- api_resource_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- kind: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- quality: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- posted: timestamp (nullable = true)
 |-- datetime_decay_updated: timestamp (nullable = true)
 |-- fees: integer (nullable = true)
 |-- seller_id: integer (nullable = true)
 |-- seller_company: string (nullable = true)
 |-- seller_realm_id: integer (nullable = true)
 |-- seller_npc: boolean (nullable = true)
 |-- snapshot_id: string (nullable = true)

+-------------------+--------------+---------------+---------+----+--------+-------+-----+--------------------------+--------------------------+-----+---------+---------------------+---------------+----------+----------------+
|captured_at        |source_page_id|api_resource_id|or

# 3. Load reference tables

In [4]:
excel_file = pd.ExcelFile(SOURCE_FILE)

reference_tables = {
    sheet_name: pd.read_excel(excel_file, sheet_name=sheet_name)
    for sheet_name in excel_file.sheet_names
}

for table_name, table_df in reference_tables.items():
    print(f"{table_name}: {len(table_df)} rows; columns={list(table_df.columns)}")


products: 151 rows; columns=['product_id', 'name', 'category']
recipes: 308 rows; columns=['product_id', 'input_product_id', 'quantity']
production: 139 rows; columns=['product_id', 'building', 'units_an_hour', 'transport_unit']
retail: 0 rows; columns=['product_id', 'quality', 'retail_phase', 'price', 'sales_time']
buildings: 45 rows; columns=['building_id', 'name', 'wage']
product_mapping: 151 rows; columns=['market_id', 'name', 'product_id', 'category', '_merge']


# 4. Reference-data validation

In [5]:
df_products_ref = reference_tables["products"]
df_mapping_ref = reference_tables["product_mapping"]
df_recipes_ref = reference_tables["recipes"]
df_production_ref = reference_tables["production"]
df_buildings_ref = reference_tables["buildings"]

# Product mapping must be one-to-one for the current model.
assert df_mapping_ref["market_id"].is_unique, "Duplicate market_id values in product_mapping."
assert df_mapping_ref["product_id"].is_unique, "Duplicate product_id values in product_mapping."

product_ids = set(df_products_ref["product_id"])
building_names = set(df_buildings_ref["name"])

assert set(df_mapping_ref["product_id"]) <= product_ids, "Mapping contains unknown product_id values."
assert set(df_products_ref["product_id"]) == set(df_mapping_ref["product_id"]), (
    "Some products have no product_mapping entry."
)
assert set(df_recipes_ref["product_id"]) <= product_ids, "Recipes contain unknown output products."
assert set(df_recipes_ref["input_product_id"]) <= product_ids, "Recipes contain unknown input products."
assert set(df_production_ref["product_id"]) <= product_ids, "Production contains unknown products."
assert set(df_production_ref["building"]) <= building_names, "Production contains unknown buildings."

print("Reference-data validation passed.")


Reference-data validation passed.


# 5. Dimension tables

In [6]:
# dim_product: product attributes + external API ID.
# Production-specific attributes (e.g. transport_unit) stay in production-related tables.
dim_product_pd = (
    df_products_ref
    .merge(
        df_mapping_ref[["product_id", "market_id"]],
        on="product_id",
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .merge(
        df_production_ref[["product_id", "transport_unit"]],
        on="product_id",
        how="left",
        validate="one_to_one"
    )
    .rename(columns={
        "market_id": "product_api_id",
        "_merge": "mapping_status",
    })
    [["product_id", "name", "category", "transport_unit", "product_api_id", "mapping_status"]]
)

dim_product_pd["product_id"] = dim_product_pd["product_id"].astype("int32")
dim_product_pd["product_api_id"] = dim_product_pd["product_api_id"].astype("int32")
dim_product_pd["mapping_status"] = dim_product_pd["mapping_status"].astype(str)
dim_product = pandas_to_spark(dim_product_pd, schema=dim_product_schema)

# dim_building
dim_building_pd = (
    reference_tables["buildings"][["building_id", "name", "wage"]]
    .copy()
    .rename(columns={"wage": "base_wage"})
)

dim_building_pd["building_id"] = dim_building_pd["building_id"].astype("int32")
dim_building_pd["base_wage"] = dim_building_pd["base_wage"].astype(float)
dim_building = pandas_to_spark(dim_building_pd, schema=dim_building_schema)

print("dim_product:", dim_product.count())
print("dim_building:", dim_building.count())


dim_product: 151
dim_building: 45


# 6. Date dimension

In [7]:
# Parse snapshot_id as UTC timestamp and derive the calendar range.
df_market_dates = df_market.withColumn(
    "snapshot_timestamp",
    F.to_timestamp("snapshot_id", "yyyyMMdd'T'HHmmss'Z'")
)

invalid_snapshot_ids = df_market_dates.filter(F.col("snapshot_timestamp").isNull()).count()
assert invalid_snapshot_ids == 0, f"Could not parse {invalid_snapshot_ids} snapshot_id values."

date_range = df_market_dates.select(
    F.min("snapshot_timestamp").alias("min_timestamp"),
    F.max("snapshot_timestamp").alias("max_timestamp"),
).first()

assert date_range["min_timestamp"] is not None, "Silver market dataset is empty."

start_date = date_range["min_timestamp"].date()
end_date = date_range["max_timestamp"].date()

dim_date_pd = pd.DataFrame({
    "date": pd.date_range(start=start_date, end=end_date, freq="D")
})
dim_date_pd["date_key"] = dim_date_pd["date"].dt.strftime("%Y%m%d").astype("int32")
dim_date_pd["year"] = dim_date_pd["date"].dt.year.astype("int32")
dim_date_pd["month"] = dim_date_pd["date"].dt.month.astype("int32")
dim_date_pd["day"] = dim_date_pd["date"].dt.day.astype("int32")
# Monday=1 ... Sunday=7
dim_date_pd["weekday"] = (dim_date_pd["date"].dt.dayofweek + 1).astype("int32")
dim_date_pd["date"] = dim_date_pd["date"].dt.date

dim_date_pd = dim_date_pd[
    ["date_key", "date", "year", "month", "day", "weekday"]
]

dim_date = pandas_to_spark(dim_date_pd, schema=dim_date_schema)

print("Date range:", start_date, "to", end_date)
print("dim_date rows:", dim_date.count())


Date range: 2026-11-04 to 2026-11-22
dim_date rows: 19


# 7. Recipe bridge

In [8]:
bridge_product_recipe_pd = (
    reference_tables["recipes"]
    [["product_id", "input_product_id", "quantity"]]
    .copy()
    .rename(columns={
        "product_id": "output_product_id",
        "quantity": "input_quantity",
    })
)

# One row = one ingredient used by one output product.
assert not bridge_product_recipe_pd.duplicated(
    ["output_product_id", "input_product_id"]
).any(), "Duplicate output-product / input-product pairs in recipes."

bridge_product_recipe_pd["output_product_id"] = bridge_product_recipe_pd["output_product_id"].astype("int32")
bridge_product_recipe_pd["input_product_id"] = bridge_product_recipe_pd["input_product_id"].astype("int32")
bridge_product_recipe_pd["input_quantity"] = bridge_product_recipe_pd["input_quantity"].astype(float)

bridge_product_recipe = pandas_to_spark(
    bridge_product_recipe_pd,
    schema=bridge_product_recipe_schema,
)

print("bridge_product_recipe rows:", bridge_product_recipe.count())


bridge_product_recipe rows: 308


# 8. Production capacity fact

In [9]:
df_production_with_building_id = df_production_ref.merge(
    df_buildings_ref[["building_id", "name"]],
    left_on="building",
    right_on="name",
    how="left",
    validate="many_to_one",
)

assert df_production_with_building_id["building_id"].notna().all(), (
    "Some production rows could not be mapped to building_id."
)

fact_production_capacity_pd = (
    df_production_with_building_id
    [["product_id", "building_id", "units_an_hour"]]
    .copy()
    .rename(columns={"units_an_hour": "base_units_per_hour"})
)

fact_production_capacity_pd["product_id"] = fact_production_capacity_pd["product_id"].astype("int32")
fact_production_capacity_pd["building_id"] = fact_production_capacity_pd["building_id"].astype("int32")
fact_production_capacity_pd["base_units_per_hour"] = fact_production_capacity_pd["base_units_per_hour"].astype(float)

fact_production_capacity = pandas_to_spark(
    fact_production_capacity_pd,
    schema=fact_production_capacity_schema,
)

print("fact_production_capacity rows:", fact_production_capacity.count())


fact_production_capacity rows: 139


# 9. Market snapshot fact

In [10]:
# Validate required Silver columns.
required_market_columns = {
    "snapshot_id", "captured_at", "api_resource_id",
    "order_id", "kind", "quantity", "quality", "price",
}
missing_columns = required_market_columns - set(df_market.columns)
assert not missing_columns, f"Missing Silver columns: {sorted(missing_columns)}"

# Create a Spark mapping table: external API resource ID -> internal product ID.
mapping_spark = spark.createDataFrame(
    df_mapping_ref[["market_id", "product_id"]].copy()
)

# The deduplication key must be present; otherwise null order IDs could collapse together.
invalid_dedupe_keys = df_market.filter(
    F.col("snapshot_id").isNull() | F.col("order_id").isNull()
).count()
assert invalid_dedupe_keys == 0, (
    f"{invalid_dedupe_keys} rows have NULL snapshot_id or order_id."
)

# Dedupe the same order within the same snapshot.
# If a duplicate has a later captured_at, keep that row.
dedupe_window = (
    Window
    .partitionBy("snapshot_id", "order_id")
    .orderBy(F.col("captured_at").desc_nulls_last())
)

df_market_deduped = (
    df_market
    .withColumn("_dedupe_rank", F.row_number().over(dedupe_window))
    .filter(F.col("_dedupe_rank") == 1)
    .drop("_dedupe_rank")
)

rows_before_dedupe = df_market.count()
rows_after_dedupe = df_market_deduped.count()
print("Rows before deduplication:", rows_before_dedupe)
print("Rows after deduplication:", rows_after_dedupe)
print("Duplicate rows removed:", rows_before_dedupe - rows_after_dedupe)

# Map the external API ID to the internal product ID.
df_market_mapped = (
    df_market_deduped
    .join(
        mapping_spark,
        df_market_deduped["api_resource_id"] == mapping_spark["market_id"],
        how="left",
    )
    .drop("market_id")
)

unmapped_market_rows = df_market_mapped.filter(F.col("product_id").isNull()).count()
assert unmapped_market_rows == 0, (
    f"{unmapped_market_rows} market rows have no matching product_mapping entry."
)

# Parse snapshot timestamp in UTC.
df_market_prepared = (
    df_market_mapped
    .withColumn(
        "snapshot_timestamp",
        F.to_timestamp("snapshot_id", "yyyyMMdd'T'HHmmss'Z'"),
    )
)

invalid_timestamps = df_market_prepared.filter(
    F.col("snapshot_timestamp").isNull()
).count()
assert invalid_timestamps == 0, f"{invalid_timestamps} rows have invalid snapshot_id values."

# Quality is numeric (0-12). Keep the source column name `quality`.
invalid_quality = df_market_prepared.filter(
    F.col("quality").isNull() |
    (F.col("quality") < 0) |
    (F.col("quality") > 12)
).count()
assert invalid_quality == 0, f"{invalid_quality} rows have quality outside 0-12 or NULL."

invalid_kind = df_market_prepared.filter(F.col("kind").isNull()).count()
assert invalid_kind == 0, f"{invalid_kind} rows have NULL kind."

# Guard against invalid quantities/prices before computing weighted metrics.
invalid_market_values = df_market_prepared.filter(
    F.col("quantity").isNull() |
    (F.col("quantity") <= 0) |
    F.col("price").isNull() |
    (F.col("price") < 0)
).count()
assert invalid_market_values == 0, (
    f"{invalid_market_values} rows have invalid quantity or price values."
)

fact_market_snapshot = (
    df_market_prepared
    .groupBy("snapshot_id", "snapshot_timestamp", "product_id", "quality", "kind")
    .agg(
        (
            F.sum(F.col("price") * F.col("quantity")) /
            F.sum("quantity")
        ).alias("avg_price"),
        F.percentile_approx("price", 0.5, 10000).alias("median_price"),
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.sum("quantity").cast("long").alias("total_quantity"),
        F.count("*").cast("long").alias("offer_count"),
    )
    .select(
        "snapshot_id",
        "snapshot_timestamp",
        "product_id",
        "quality",
        "kind",
        "avg_price",
        "median_price",
        "min_price",
        "max_price",
        "total_quantity",
        "offer_count",
    )
)

# Apply the explicit Gold schema's column names and data types.
fact_market_snapshot = fact_market_snapshot.select(*[
    F.col(field.name).cast(field.dataType).alias(field.name)
    for field in fact_market_snapshot_schema.fields
])

print("fact_market_snapshot rows:", fact_market_snapshot.count())
fact_market_snapshot.orderBy(
    "snapshot_id", "product_id", "quality", "kind"
).show(10, truncate=False)


Rows before deduplication: 39208
Rows after deduplication: 39208
Duplicate rows removed: 0
fact_market_snapshot rows: 4813
+----------------+-------------------+----------+-------+----+------------------+------------+---------+---------+--------------+-----------+
|snapshot_id     |snapshot_timestamp |product_id|quality|kind|avg_price         |median_price|min_price|max_price|total_quantity|offer_count|
+----------------+-------------------+----------+-------+----+------------------+------------+---------+---------+--------------+-----------+
|20261104T130919Z|2026-11-04 13:09:19|2         |0      |3   |2.578840472287841 |2.6         |2.3      |2.7      |2489160       |178        |
|20261104T130919Z|2026-11-04 13:09:19|2         |1      |3   |2.565646069677463 |2.55        |2.55     |2.6      |471946        |13         |
|20261104T130919Z|2026-11-04 13:09:19|2         |2      |3   |2.659506234978086 |2.65        |2.5      |2.7      |884125        |9          |
|20261104T130919Z|2026-11

# 10. Gold validation

In [11]:
gold_tables = {
    "dim_product": dim_product,
    "dim_building": dim_building,
    "dim_date": dim_date,
    "bridge_product_recipe": bridge_product_recipe,
    "fact_production_capacity": fact_production_capacity,
    "fact_market_snapshot": fact_market_snapshot,
}

for table_name, table_df in gold_tables.items():
    print(f"\n{table_name}")
    table_df.printSchema()
    print("Rows:", table_df.count())

# Basic key / grain checks.
assert dim_product.select("product_id").distinct().count() == dim_product.count()
assert dim_building.select("building_id").distinct().count() == dim_building.count()
assert dim_date.select("date_key").distinct().count() == dim_date.count()

assert (
    bridge_product_recipe
    .groupBy("output_product_id", "input_product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
) == 0

assert (
    fact_market_snapshot
    .groupBy("snapshot_id", "product_id", "quality", "kind")
    .count()
    .filter(F.col("count") > 1)
    .count()
) == 0

print("\nGold validation passed.")



dim_product
root
 |-- product_id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- category: string (nullable = true)
 |-- transport_unit: double (nullable = true)
 |-- product_api_id: integer (nullable = true)
 |-- mapping_status: string (nullable = false)

Rows: 151

dim_building
root
 |-- building_id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- base_wage: double (nullable = true)

Rows: 45

dim_date
root
 |-- date_key: integer (nullable = false)
 |-- date: date (nullable = false)
 |-- year: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- day: integer (nullable = false)
 |-- weekday: integer (nullable = false)

Rows: 19

bridge_product_recipe
root
 |-- output_product_id: integer (nullable = false)
 |-- input_product_id: integer (nullable = false)
 |-- input_quantity: double (nullable = false)

Rows: 308

fact_production_capacity
root
 |-- product_id: integer (nullable = false)
 |-- building_id: integer (nullable = f

# 11. Save Gold tables as Parquet

In [12]:
def save_gold_table(df, table_name: str, gold_path: Path):
    """Overwrite one Gold table directory with a Spark DataFrame in Parquet format."""
    output_path = gold_path / table_name

    (
        df.write
        .mode("overwrite")
        .parquet(str(output_path))
    )

    print(f"Saved: {output_path}")
    print(f"Rows: {df.count()}")


for table_name, table_df in gold_tables.items():
    save_gold_table(table_df, table_name, GOLD_PATH)


Saved: c:\Projekty\SimCompanies\data\gold\dim_product
Rows: 151
Saved: c:\Projekty\SimCompanies\data\gold\dim_building
Rows: 45
Saved: c:\Projekty\SimCompanies\data\gold\dim_date
Rows: 19
Saved: c:\Projekty\SimCompanies\data\gold\bridge_product_recipe
Rows: 308
Saved: c:\Projekty\SimCompanies\data\gold\fact_production_capacity
Rows: 139
Saved: c:\Projekty\SimCompanies\data\gold\fact_market_snapshot
Rows: 4813


# 12. Read-back check

In [13]:
# Verify that all six output directories can be read back.
for table_name in gold_tables:
    saved_df = spark.read.parquet(str(GOLD_PATH / table_name))
    print(f"{table_name}: {saved_df.count()} rows")


dim_product: 151 rows
dim_building: 45 rows
dim_date: 19 rows
bridge_product_recipe: 308 rows
fact_production_capacity: 139 rows
fact_market_snapshot: 4813 rows
